    #   Pre-requisites

In [1]:
import sys
# Define the path to the directory containing the module
module_dir = "../Main/RequiredFuntions"

# Append the directory to sys.path
sys.path.append(module_dir)

In [2]:
import json
import time
import hashlib
import threading
import functions as fn
import dataTransfer as DT
import InitializationPhase as IP
import EncryptionDecryption as ED

    #   ACK-1

In [3]:
img_path = "../Main/RequiredData/Fingerprint/12.jpg"
model_path = "../Main/Model/Accelerometer"
accel_data = "../Main/RequiredData/Accelerometer/testingdataM55.csv"

In [4]:
# GENERATING PUBLIC AND PRIVATE KEYS FOR GATEWAY
GATEWAY = IP.ECCPublicPrivateKeyGeneration()

DEVICE = IP.ECCPublicPrivateKeyGeneration()

In [48]:
with open('../Main/ReceivedData/keys.json', 'r') as file:
    keys = json.load(file)

In [50]:
keys["device_public"] = DEVICE[0].decode()
keys["device_private"] = DEVICE[1].decode()

In [53]:
with open(f'./ReceivedData/keys.json', 'w') as json_file:
        json.dump(keys, json_file, indent=4)

In [52]:
with open('../Main/ReceivedData/presharedKeys.json', 'r') as file:
    presharedkey = json.load(file)

presharedkey = presharedkey["key"]

In [6]:
deviceUniqueId_j = "RajeshHomeDevice1"
deviceNonce_j = fn.nonce_gen()

In [7]:
encryptedUniqueId = ED.symmetric_key_encryption(deviceUniqueId_j, 
                                                  "".encode(), 
                                                  presharedkey)

In [8]:
compute_j = hashlib.sha256((deviceUniqueId_j + str(deviceNonce_j) + presharedkey).encode()).hexdigest()

In [9]:
DeviceData = {
    "encryptedUniqueId" : encryptedUniqueId,
    "deviceNonce" : deviceNonce_j,
    "compute" : compute_j
}

In [10]:
# Create and start threads
thread1 = threading.Thread(target=DT.receive, args=("device_received_data.json",))  # Start the receive function
thread2 = threading.Thread(target=DT.send, args=(DeviceData,))  # Start the send function with arguments

thread1.start()
time.sleep(2)  # Ensure the server starts before sending
thread2.start()

# Wait for threads to complete
thread1.join()
thread2.join()

print("Data transfer and storage complete.")

Server is listening on port 12345...
Connected byData transfer done.
 ('127.0.0.1', 58572)
Received data saved to 'received_data.json'.
Data transfer and storage complete.


In [11]:
# step 2
print("Starting Step 2")
json_data = fn.read_json_file("../Main/ReceivedData/device_received_data.json")

Starting Step 2


In [12]:
decryptedId = ED.symmetric_key_decryption(json_data["encryptedUniqueId"], 
                                                  presharedkey)

In [13]:
decryptedId

(b'RajeshHomeDevice1', b'')

In [14]:
compute_j = hashlib.sha256((deviceUniqueId_j + str(deviceNonce_j) + presharedkey).encode()).hexdigest()

In [15]:
validationCompute_j = hashlib.sha256((decryptedId[0].decode() + 
                                      str(json_data["deviceNonce"]) +
                                      presharedkey).encode()).hexdigest()

In [16]:
validationCompute_j == compute_j

True

In [17]:
gatewayUniqueId_j = "rajeshGateway"

In [18]:
fn.AddNewDevice(1, "defaultdevice1", "../Main/DB/HomeDeviceList.hdf5")

🎯 Encrypted Data Saved for Training ID `1` in ../Main/DB/HomeDeviceList.hdf5


'success'

In [19]:
# creating devicelist for default device for training (assuming already a device just for training purposes)
NewDevice = fn.GetNewDeviceID(fn.load_all_data("../Main/DB/HomeDeviceList.hdf5"))
fn.AddNewDevice(NewDevice, decryptedId[0].decode(), "../Main/DB/HomeDeviceList.hdf5")

sorted_data = 1
🎯 Encrypted Data Saved for Training ID `2` in ../Main/DB/HomeDeviceList.hdf5


'success'

In [20]:
hashedGatewayID = hashlib.sha256(gatewayUniqueId_j.encode()).hexdigest()
hashedUniqueId_j = hashlib.sha256(decryptedId[0]).hexdigest()

In [21]:
#generate nounce in gateway
gatewayNonce_j = fn.nonce_gen()

Secret_j = int(gatewayNonce_j) ^ int(json_data["deviceNonce"])

In [22]:
# shares generated by gateway
generatedRandomNumber = int(fn.nonce_gen())
sharesGeneratedByGateway = (Secret_j + (2 * generatedRandomNumber)) % fn.primeNumbergenerator()

In [54]:
sharesGeneratedByGateway

30

In [23]:
secretIntegrity_j = hashlib.sha256(
    (hashlib.sha256((str(Secret_j) + decryptedId[0].decode()).encode()).hexdigest() + str(json_data["deviceNonce"])).encode()).hexdigest()

In [24]:
# temporal identity to encrypt R and gatwayNonce

temporalIdentity_j = hashlib.sha256((decryptedId[0].decode() + str(json_data["deviceNonce"]) + DEVICE[0].decode()).encode()).hexdigest()

In [25]:
encryptedRandomNumber = fn.xor_strings(str(generatedRandomNumber), temporalIdentity_j)
encryptedRandomNumber = fn.xor_strings(encryptedRandomNumber, presharedkey)

encryptedGatewayNonce = fn.xor_strings(str(gatewayNonce_j), temporalIdentity_j)
encryptedGatewayNonce = fn.xor_strings(encryptedGatewayNonce, presharedkey)

In [26]:
gatewaySecondNonce_j = fn.nonce_gen()

temporalIdentityGateway = hashlib.sha256((gatewayUniqueId_j + GATEWAY[0].decode() + str(gatewaySecondNonce_j)).encode()).hexdigest()

compute_G = hashlib.sha256((
    temporalIdentityGateway + 
    secretIntegrity_j + 
    encryptedRandomNumber + 
    encryptedGatewayNonce + 
    str(gatewaySecondNonce_j)).encode()).hexdigest()

In [27]:
# saving the necesssay values
fn.gatewayStore(decryptedId[0].decode(), hashlib.sha256((str(Secret_j) + decryptedId[0].decode()).encode()).hexdigest(), sharesGeneratedByGateway)

updated gateway storage


In [28]:
gatewayJson = {
    'secretIntegrity' : secretIntegrity_j,
    'encryptedRandomNumber' : encryptedRandomNumber,
    'encryptedGatewayNonce' : encryptedGatewayNonce,
    'gatwayNonce' : gatewaySecondNonce_j,
    'computeG' : compute_G
}

In [29]:
# Create and start threads
thread1 = threading.Thread(target=DT.receive, args=('received_data_device_gateway.json',))  # Start the receive function
thread2 = threading.Thread(target=DT.send, args=(gatewayJson,))  # Start the send function with arguments

thread1.start()
time.sleep(2)  # Ensure the server starts before sending
thread2.start()

# Wait for threads to complete
thread1.join()
thread2.join()

print("Data transfer and storage complete.")

Server is listening on port 12345...
Connected byData transfer done.
 ('127.0.0.1', 58576)
Received data saved to 'received_data.json'.
Data transfer and storage complete.


In [30]:
# step 3
print("Starting Step 3")
gatewayJsonData = fn.read_json_file("../Main/ReceivedData/received_data_device_gateway.json")

Starting Step 3


In [31]:
#compute temporal identity user
temporalIdentity_j_s3 = hashlib.sha256((deviceUniqueId_j + str(deviceNonce_j) + DEVICE[0].decode() ).encode()).hexdigest()

#compute temporal indentity gateway
temporalIdentity_j_g_s3 = hashlib.sha256((gatewayUniqueId_j + GATEWAY[0].decode() + str(gatewayJsonData['gatwayNonce'])).encode()).hexdigest()

In [32]:
validateComputeG = hashlib.sha256((temporalIdentity_j_g_s3 + 
                    gatewayJsonData['secretIntegrity'] + 
                    str(gatewayJsonData['encryptedRandomNumber']) + 
                    str(gatewayJsonData['encryptedGatewayNonce']) + 
                    str(gatewayJsonData['gatwayNonce'])).encode()).hexdigest()

In [33]:
# unencrypt randomnumber and gateway nonce
GatewayRandomNumber = fn.xor_strings(gatewayJsonData['encryptedRandomNumber'], temporalIdentity_j_s3)
GatewayRandomNumber = fn.xor_strings(GatewayRandomNumber, presharedkey)

GatewayGeneratedNonce = fn.xor_strings(str(gatewayJsonData['encryptedGatewayNonce']), temporalIdentity_j_s3)
GatewayGeneratedNonce = fn.xor_strings(GatewayGeneratedNonce, presharedkey)

In [34]:
secretComputed = int(deviceNonce_j) ^ int(GatewayGeneratedNonce)
userShare = (secretComputed + int(GatewayRandomNumber)) % fn.primeNumbergenerator()

In [35]:
validationSecretIntegrity = hashlib.sha256((str(secretComputed) + deviceUniqueId_j).encode()).hexdigest()
validationSecretIntegrity = hashlib.sha256( (validationSecretIntegrity + str(deviceNonce_j)).encode()).hexdigest()

In [36]:
print(f"Secret Integrity validation : {validationSecretIntegrity == secretIntegrity_j}")

Secret Integrity validation : True


In [37]:
with open('../Main/ReceivedData/datastore.json', 'r') as file:
    datastore = json.load(file)

In [ ]:
datastore[gatewayUniqueId_j]["smartDevices"] = {
    deviceUniqueId_j: {
        "secretIntegrity" : validationSecretIntegrity,
        "secretComputed" : secretComputed,
        "sharesGeneratedByGateway": sharesGeneratedByGateway
    }
}

In [41]:
with open(f'./ReceivedData/datastore.json', 'w') as json_file:
        json.dump(datastore, json_file, indent=4)